In [52]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter("ignore")
from sklearn.linear_model import LogisticRegression
from sklearn import tree 
from sklearn import ensemble 
from sklearn import metrics 
import optuna
from catboost import CatBoostClassifier
from sklearn.model_selection import cross_val_score

In [53]:
#Прочитаем обработанные данные
X_train = pd.read_csv('../data/X_train_scaled.csv')
X_test = pd.read_csv('../data/X_test_scaled.csv')

y_train = pd.read_csv('../data/y_train.csv').squeeze()
y_test = pd.read_csv('../data/y_test.csv').squeeze()

In [54]:
#Инициализируем функцию для подсчета метрик моделей
results = []

def evaluate_model(name, model, X_test, y_test):
    
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:,1]
    
    results.append({
        'model': name,
        'accuracy': metrics.accuracy_score(y_test, y_pred),
        'precision': metrics.precision_score(y_test, y_pred),
        'f1': metrics.f1_score(y_test, y_pred),
        'roc_auc': metrics.roc_auc_score(y_test, y_proba)
    })

In [55]:
#Попробуем построить рандомный лес
#Инициализируем лес
rf = ensemble.RandomForestClassifier(
    n_estimators=100,
    criterion='gini',
    min_samples_leaf=5,
    max_depth=10,
    random_state=42
)
#Обучаем лес
rf.fit(X_train, y_train)
evaluate_model('RandomForest', rf, X_test, y_test)
#Делаем предсказание
y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

print('Accuracy:', round(metrics.accuracy_score(y_test, y_pred), 3))
print('Precision:', round(metrics.precision_score(y_test, y_pred), 3))
print('F1:', round(metrics.f1_score(y_test, y_pred), 3))
print('ROC-AUC:', round(metrics.roc_auc_score(y_test, y_proba), 3))

Accuracy: 0.692
Precision: 0.712
F1: 0.63
ROC-AUC: 0.746


Случайный лес справляется с задачей лучше, чем модели в бейзлайне, попробуем градиентный бустинг

In [56]:
#Инициализируем градиентный бустинг
gb = ensemble.GradientBoostingClassifier(
    learning_rate=0.05,
    n_estimators=300,
    min_samples_leaf=5,
    max_depth=5,
    random_state=42
)
#Обучаем бустинг
gb.fit(X_train, y_train)
evaluate_model('Boosting', gb, X_test, y_test)
#Делаем предсказание
y_pred = gb.predict(X_test)
y_proba = gb.predict_proba(X_test)[:, 1]

print('Accuracy:', round(metrics.accuracy_score(y_test, y_pred), 3))
print('Precision:', round(metrics.precision_score(y_test, y_pred), 3))
print('F1:', round(metrics.f1_score(y_test, y_pred), 3))
print('ROC-AUC:', round(metrics.roc_auc_score(y_test, y_proba), 3))

Accuracy: 0.689
Precision: 0.704
F1: 0.627
ROC-AUC: 0.742


Совсем немного, но хуже, попробуем стекинг с логрегрессией, деревом и бустингом

In [57]:
#Инициализируем регрессию
lr = LogisticRegression(
    solver='sag',
    random_state=42,
    max_iter=1000
)
#Инициализируем дерево
tree_model = tree.DecisionTreeClassifier(
    random_state=42,
    criterion='entropy'
)

In [58]:
#Делаем стэккинг с мета-моделью логистической регрессией
stack = ensemble.StackingClassifier(
    estimators=[
        ('lr', lr),
        ('tree', tree_model),
        ('gb', gb)
    ],
    final_estimator=LogisticRegression()
)
#Обучаем стеккинг
stack.fit(X_train, y_train)
evaluate_model('Stacking', stack, X_test, y_test)
#делаем предсказание
y_pred = stack.predict(X_test)
y_proba = stack.predict_proba(X_test)[:, 1]

print('Accuracy:', round(metrics.accuracy_score(y_test, y_pred), 3))
print('Precision:', round(metrics.precision_score(y_test, y_pred), 3))
print('F1:', round(metrics.f1_score(y_test, y_pred), 3))
print('ROC-AUC:', round(metrics.roc_auc_score(y_test, y_proba), 3))

Accuracy: 0.688
Precision: 0.694
F1: 0.633
ROC-AUC: 0.746


Практически без изменений

In [59]:
importances = gb.feature_importances_
feature_importance = pd.Series(
    gb.feature_importances_,
    index=X_train.columns
)

feature_importance.sort_values(ascending=False).head(3)

13    0.249454
0     0.230470
8     0.096655
dtype: float64

Попробуем найти гиперпараметры с помощью ***optuna***

In [60]:
def optuna_rf(trial):

    n_estimators = trial.suggest_int('n_estimators', 100, 200)
    max_depth = trial.suggest_int('max_depth', 10, 30)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 2, 10)

    model = ensemble.RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        random_state=42,
        n_jobs=-1
    )

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring='f1',
        n_jobs=-1
    )

    return scores.mean()

In [61]:
study = optuna.create_study(direction='maximize')
study.optimize(optuna_rf, n_trials=50)

[I 2026-03-07 17:18:21,004] A new study created in memory with name: no-name-d1ef7af5-8711-4b11-9870-cfbdd51473a8
[I 2026-03-07 17:18:22,405] Trial 0 finished with value: 0.6334032449221367 and parameters: {'n_estimators': 112, 'max_depth': 24, 'min_samples_leaf': 5}. Best is trial 0 with value: 0.6334032449221367.
[I 2026-03-07 17:18:23,626] Trial 1 finished with value: 0.6387495067161785 and parameters: {'n_estimators': 185, 'max_depth': 15, 'min_samples_leaf': 6}. Best is trial 1 with value: 0.6387495067161785.
[I 2026-03-07 17:18:24,837] Trial 2 finished with value: 0.6289094042600352 and parameters: {'n_estimators': 183, 'max_depth': 11, 'min_samples_leaf': 10}. Best is trial 1 with value: 0.6387495067161785.
[I 2026-03-07 17:18:26,059] Trial 3 finished with value: 0.6281642130273427 and parameters: {'n_estimators': 186, 'max_depth': 14, 'min_samples_leaf': 9}. Best is trial 1 with value: 0.6387495067161785.
[I 2026-03-07 17:18:27,196] Trial 4 finished with value: 0.62901785117709

In [62]:
print(study.best_params)

{'n_estimators': 200, 'max_depth': 10, 'min_samples_leaf': 3}


In [63]:
best_rf = ensemble.RandomForestClassifier(
    **study.best_params,
    random_state=42,
    n_jobs=-1
)

best_rf.fit(X_train, y_train)
evaluate_model('best_rf', best_rf, X_test, y_test)
y_pred = best_rf.predict(X_test)
y_proba = best_rf.predict_proba(X_test)[:, 1]

print('Accuracy:', round(metrics.accuracy_score(y_test, y_pred), 3))
print('Precision:', round(metrics.precision_score(y_test, y_pred), 3))
print('F1:', round(metrics.f1_score(y_test, y_pred), 3))
print('ROC-AUC:', round(metrics.roc_auc_score(y_test, y_proba), 3))

Accuracy: 0.693
Precision: 0.71
F1: 0.632
ROC-AUC: 0.746


In [64]:
cat = CatBoostClassifier(
    depth=6,
    learning_rate=0.05,
    iterations=500,
    random_state=42,
    verbose=False
)

cat.fit(X_train, y_train)
evaluate_model('CatBoost', cat, X_test, y_test)
pred = cat.predict(X_test)
y_proba = cat.predict_proba(X_test)[:, 1]

print('Accuracy:', round(metrics.accuracy_score(y_test, y_pred), 3))
print('Precision:', round(metrics.precision_score(y_test, y_pred), 3))
print('F1:', round(metrics.f1_score(y_test, y_pred), 3))
print('ROC-AUC:', round(metrics.roc_auc_score(y_test, y_proba), 3))

Accuracy: 0.693
Precision: 0.71
F1: 0.632
ROC-AUC: 0.744


In [65]:
def objective(trial):
    params = {
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'iterations': trial.suggest_int('iterations', 200, 1000),
        'l2_leaf_reg': trial.suggest_int('l2_leaf_reg', 1, 10),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 20),
        'random_state': 42,
        'verbose': False
    }

    model = CatBoostClassifier(**params)

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring='f1',
        n_jobs=-1
    )

    return scores.mean()

In [66]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

[I 2026-03-07 17:18:38,524] A new study created in memory with name: no-name-e8e42892-14ae-45c4-ac47-796fa66d39ab
[I 2026-03-07 17:18:41,873] Trial 0 finished with value: 0.6377163853339789 and parameters: {'depth': 9, 'learning_rate': 0.0239501570633012, 'iterations': 655, 'l2_leaf_reg': 10, 'min_data_in_leaf': 19}. Best is trial 0 with value: 0.6377163853339789.
[I 2026-03-07 17:18:51,881] Trial 1 finished with value: 0.6172570947972501 and parameters: {'depth': 10, 'learning_rate': 0.16336634492435234, 'iterations': 989, 'l2_leaf_reg': 7, 'min_data_in_leaf': 13}. Best is trial 0 with value: 0.6377163853339789.
[I 2026-03-07 17:18:53,354] Trial 2 finished with value: 0.6343567349387719 and parameters: {'depth': 5, 'learning_rate': 0.0765533141263439, 'iterations': 804, 'l2_leaf_reg': 5, 'min_data_in_leaf': 7}. Best is trial 0 with value: 0.6377163853339789.
[I 2026-03-07 17:18:54,321] Trial 3 finished with value: 0.6355978435641141 and parameters: {'depth': 6, 'learning_rate': 0.0415

In [67]:
print('Best params:', study.best_params)
print('Best F1:', study.best_value)

Best params: {'depth': 7, 'learning_rate': 0.013014539106538102, 'iterations': 742, 'l2_leaf_reg': 3, 'min_data_in_leaf': 19}
Best F1: 0.6398529411048633


In [68]:
best_cat = CatBoostClassifier(
    **study.best_params,
    random_state=42,
    verbose=False
)

best_cat.fit(X_train, y_train)
evaluate_model('BestCat', best_cat, X_test, y_test)

y_pred = best_cat.predict(X_test)

acc = metrics.accuracy_score(y_test, y_pred)
f1 = metrics.f1_score(y_test, y_pred)

print('Accuracy:', round(acc, 3))
print('F1:', round(f1, 3))

Accuracy: 0.696
F1: 0.628


In [69]:
results_df = pd.DataFrame(results)

results_df.round(3).sort_values(by='f1', ascending=False)

,model,accuracy,precision,f1,roc_auc
2,Stacking,0.688,0.694,0.633,0.746
3,best_rf,0.693,0.710,0.632,0.746
0,RandomForest,0.692,0.712,0.630,0.746
5,BestCat,0.696,0.724,0.628,0.749
1,Boosting,0.689,0.704,0.627,0.742
4,CatBoost,0.688,0.703,0.627,0.744
